In [ ]:
import numpy as np
import pandas as pd
import wfdb
from pathlib import Path
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, find_peaks

np.random.seed(42)

# Paths
ROOT = Path("../../").resolve()

DB_DIR = ROOT / "db"
MIMICIV_DIR = DB_DIR / "mimiciv"
MIMICIV_ECG_DIR = DB_DIR / "mimiciv_ecg"
PROCESSED_DIR = DB_DIR / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("MIMIC-IV:", MIMICIV_DIR)
print("MIMIC-IV-ECG:", MIMICIV_ECG_DIR)
print("Processed:", PROCESSED_DIR)

In [ ]:
# Load clinical tables

patients = pd.read_csv(list(MIMICIV_DIR.rglob("patients.csv"))[0])
admissions = pd.read_csv(list(MIMICIV_DIR.rglob("admissions.csv"))[0])
diagnoses = pd.read_csv(list(MIMICIV_DIR.rglob("diagnoses_icd.csv"))[0])

# Optional ICU filtering
icustays_path = list(MIMICIV_DIR.rglob("icustays.csv"))
icustays = pd.read_csv(icustays_path[0]) if icustays_path else None

# Normalize columns
patients.columns = patients.columns.str.lower()
admissions.columns = admissions.columns.str.lower()
diagnoses.columns = diagnoses.columns.str.lower()

if icustays is not None:
    icustays.columns = icustays.columns.str.lower()

# Stroke labels
stroke_codes = ["I60","I61","I62","I63","I64"]

diagnoses["stroke"] = diagnoses["icd_code"].astype(str).str.startswith(tuple(stroke_codes))

stroke_labels = (
    diagnoses.groupby("subject_id")["stroke"]
    .max()
    .reset_index()
)

print("Stroke rate:", stroke_labels.stroke.mean())

# ICU → regular person filtering
if icustays is not None:

    icu_summary = (
        icustays.groupby("subject_id")
        .agg(
            icu_los=("los","mean")
        )
        .reset_index()
    )

    # keep lower severity patients
    icu_summary = icu_summary[
        icu_summary["icu_los"] < 2
    ]

    keep_subjects = set(icu_summary.subject_id)

else:
    keep_subjects = set(stroke_labels.subject_id)

print("Subjects kept:", len(keep_subjects))

In [ ]:
record_list = list(MIMICIV_ECG_DIR.rglob("record_list.csv"))[0]
ecg_index = pd.read_csv(record_list)

ecg_index.columns = ecg_index.columns.str.lower()

ecg_index = ecg_index[
    ecg_index.subject_id.isin(keep_subjects)
]

print("ECG records:", len(ecg_index))

def bandpass_filter(ecg, fs=250, low=5, high=15):
    b, a = butter(1, [low/(fs/2), high/(fs/2)], btype='band')
    return filtfilt(b, a, ecg)

def fast_rpeak_detector(ecg, fs=250):
    x = bandpass_filter(ecg, fs)
    x2 = x * x
    win = int(0.15 * fs)
    mwa = np.convolve(x2, np.ones(win)/win, mode='same')
    peaks, _ = find_peaks(mwa, distance=int(0.25 * fs))
    return peaks

def fast_hr_hrv_from_rpeaks(rpeaks, fs=250, window_sec=60, step_sec=60):
    """
    Extract HR and HRV (RMSSD) from R-peaks using a sliding window.
    step_sec < window_sec creates overlapping windows = more samples.
    """
    if len(rpeaks) < 3:
        return None, None

    rr_intervals = np.diff(rpeaks) / fs
    rr_intervals = rr_intervals[(rr_intervals > 0.3) & (rr_intervals < 2.0)]

    if len(rr_intervals) < 3:
        return None, None

    rr_times = (rpeaks[:-1] + np.diff(rpeaks) / 2) / fs
    t_max = rr_times[-1]
    times = np.arange(0, t_max - window_sec + 1, step_sec)

    hr = np.full(len(times), np.nan, dtype="float32")
    hrv = np.full(len(times), np.nan, dtype="float32")

    for i, t0 in enumerate(times):
        t1 = t0 + window_sec
        mask = (rr_times >= t0) & (rr_times < t1)
        if np.sum(mask) < 3:
            continue
        rr_win = rr_intervals[mask]
        hr_val = 60.0 / np.mean(rr_win)
        if 30 <= hr_val <= 220:
            hr[i] = hr_val
        diff_rr = np.diff(rr_win)
        if len(diff_rr) > 0:
            hrv[i] = np.sqrt(np.mean(diff_rr**2))

    valid = ~np.isnan(hr)
    if np.sum(valid) < 10:
        return None, None

    return hr[valid], hrv[valid]

def extract_hr_fast(ecg, fs=250):
    rpeaks = fast_rpeak_detector(ecg, fs)
    hr, _ = fast_hr_hrv_from_rpeaks(rpeaks, fs)
    return hr

windows = []

for _, row in ecg_index.head(5000).iterrows():

    try:
        path = MIMICIV_ECG_DIR / row["path"]

        record = wfdb.rdrecord(str(path))
        ecg = record.p_signal

        hr = extract_hr_fast(ecg)

        windows.append({
            "subject_id": row.subject_id,
            "hr": hr
        })

    except Exception:
        continue

windows = pd.DataFrame(windows)

print("Windows built:", len(windows))

In [ ]:
def dropout(x, p=0.3):
    mask = np.random.rand(len(x)) > p
    return x * mask


def noise(x):
    return x + np.random.normal(0,1,len(x))


windows["hr"] = windows["hr"].apply(dropout)
windows["hr"] = windows["hr"].apply(noise)

dataset = windows.merge(
    stroke_labels,
    on="subject_id",
    how="left"
)

dataset["stroke"] = dataset["stroke"].fillna(0)

dataset.to_parquet(
    PROCESSED_DIR / "stroke_wearable_dataset.parquet"
)

print("Saved dataset")
print(dataset.head())

print("Shape:", dataset.shape)
print("Stroke rate:", dataset.stroke.mean())